In [3]:

import pandas as pd
import plotly.express as px


In [21]:

# Which top and bottom 5 EU countries have the most and least train freight?

import pandas as pd
import plotly.express as px

# Loading the dataset
df = pd.read_csv("../Processed/merged_eurostat_clean_V2.csv")

# Renaming columns for clarity
df = df.rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "NST_TOTAL_MIO_TKM": "million_tkm"
})

# Converting freight column to numeric
df["million_tkm"] = pd.to_numeric(df["million_tkm"], errors="coerce")

# Summing total rail freight by country over all years
total_freight = df.groupby("country")["million_tkm"].sum().reset_index()

# Sorting and selecting Top 10
top10 = total_freight.sort_values(by="million_tkm", ascending=False).head(10)

# Plotting using Plotly
fig = px.bar(
    top10,
    x="country",
    y="million_tkm",
    title="Top 10 EU Countries by Total Rail Freight (Million tonne-km)",
    labels={"country": "Country", "million_tkm": "Total Rail Freight (Million tonne-km)"},
    text="million_tkm",
    color="million_tkm",
    color_continuous_scale="Blues"
)

fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.update_layout(
    xaxis_tickangle=0,
    plot_bgcolor="white",
    font=dict(size=12),
    showlegend=False
)

fig.show()



The bar chart shows that rail freight activity in the EU is highly concentrated among a few key countries. Germany dominates rail freight transport by a large margin, carrying nearly 2 million million tonne-km, followed by Poland and France. These three countries alone account for a substantial share of total EU rail freight, reflecting their large economies, strategic logistics hubs and strong rail infrastructure.

Sweden, Austria, Czechia, Latvia, Lithuania, Romania, and Switzerland also appear within the top 10, but with significantly smaller volumes compared to Germany. Many of these countries play vital roles in transit freight corridors, especially those positioned along major European east-west and north-south transport routes.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Top 5
top5 = total_freight.sort_values(by="million_tkm", ascending=False).head(5)

# Bottom 5
bottom5 = (
    total_freight.sort_values(by="million_tkm", ascending=True)
    .head(5)
    .sort_values(by="million_tkm", ascending=False)
)

# Subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Top 5 EU Countries by Rail Freight", "Bottom 5 EU Countries by Rail Freight")
)

# Top 5 Bar Chart
fig.add_trace(
    go.Bar(
        x=top5["country"],
        y=top5["million_tkm"],
        marker_color="steelblue"
    ),
    row=1, col=1
)

# Bottom 5 Bar Chart
fig.add_trace(
    go.Bar(
        x=bottom5["country"],
        y=bottom5["million_tkm"],
        marker_color="salmon"
    ),
    row=1, col=2
)

# Layout Formatting
fig.update_layout(
    title="Top vs Bottom 5 EU Countries by Total Rail Freight (Million tkm)",
    plot_bgcolor="white",
    showlegend=False,
    height=500
)


fig.update_xaxes(title_text="Country", tickangle=0, row=1, col=1)
fig.update_xaxes(title_text="Country", tickangle=0, row=1, col=2)

fig.update_yaxes(title_text="Total Rail Freight (Million tkm)", row=1, col=1)

fig.show()



This comparison highlights the stark difference in rail freight volumes between the largest and smallest contributors in the EU rail network. Germany again stands out as the dominant player, followed by Poland, France, Sweden, and Austria. These countries have strong industrial bases, major logistics hubs, and well-developed freight rail corridors, which explains their high rail freight volumes.

In contrast, the bottom 5 countries — Denmark, Belgium, Greece, Luxembourg, and Ireland — contribute significantly less to total EU rail freight. Smaller land area, lower industrial output, limited freight-rail networks, or greater reliance on maritime/road transport can explain these low volumes. For example, Ireland’s freight rail network is minimal and largely focused on passenger transport, while Luxembourg’s small territory naturally limits freight movements

In [14]:
# Analyzing freight trends over time for the top 5 countries
top5 = total_freight.sort_values(by="million_tkm", ascending=False).head(5)["country"].tolist()
df_top5 = df[df["country"].isin(top5)]
df_top5_yearly = df_top5.groupby(["year","country"])["million_tkm"].sum().reset_index()

# Plotly line chart
fig = px.line(
    df_top5_yearly,
    x="year",
    y="million_tkm",
    color="country",
    title="Freight Trend Over Time: Top 5 EU Countries",
    markers=True
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Rail Freight (million tkm)",
    plot_bgcolor="white",
    legend_title="Country",
    font=dict(size=12),
)

fig.update_xaxes(showgrid=True, gridwidth=0.3, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridwidth=0.3, gridcolor="lightgrey")

fig.show()


This time-series chart illustrates the evolution of rail freight volumes among the top five EU rail freight countries: Germany, Poland, France, Sweden, and Austria. Across the time period, all countries show relatively stable freight flows with mild fluctuations, indicating long-term resilience in rail-based logistics.

Germany consistently dominates the EU market, maintaining the highest freight levels throughout. Its trend displays slight dips in certain years (e.g., around 2009 and 2020), reflecting broader economic shocks such as the financial crisis and the COVID-19 pandemic, but it quickly rebounds each time. Poland shows steady long-term growth, underscoring its increasing role as a key freight corridor between Western Europe and Eastern markets. France, Sweden, and Austria exhibit more stable and moderate freight volumes, reinforcing their position as established, well-integrated freight actors within the EU network.

A notable anomaly appears around 2023 for Poland, where recorded freight values drop sharply before returning to prior levels the following year. This may reflect temporary reporting anomalies or a one-year shock (e.g., energy crisis effects or border bottlenecks).

In [20]:
from scipy.stats import chisquare
# Use total_freight to build the country-value series already computed earlier
total_sorted = total_freight.set_index("country")["million_tkm"].sort_values(ascending=False)

# Observed,expected values
observed = total_sorted.values
expected = [observed.mean()] * len(observed)

# Chi-square test
chi_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square Statistic:", chi_stat)
print("p-value:", p_value)

# Plotly bar chart
fig = go.Figure()

# Observed freight bars
fig.add_trace(go.Bar(
    x=total_sorted.index,
    y=total_sorted.values,
    marker_color="steelblue"
))

# Expected equal-share horizontal line
fig.add_hline(
    y=observed.mean(),
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text="Expected Value",
    annotation_position="top left"
)

# Layout
fig.update_layout(
    title="Distribution of Rail Freight Across EU Countries (Chi-Square Test)",
    xaxis_title="Country",
    yaxis_title="Total Rail Freight (million tkm)",
    xaxis_tickangle=0,
    plot_bgcolor="white",
    showlegend=False,
    height=500
)

fig.show()


Chi-square Statistic: 15980911.611509945
p-value: 0.0


This chart visualizes the total rail freight volume for each EU country, compared against an “equal-share” benchmark (red dashed line). If rail freight were evenly distributed, all countries would align near the expected value line. Instead, the bars show extreme concentration in a few economies.

Germany alone far exceeds all others, followed by Poland and France. These countries act as the core freight corridors of Europe, supported by large industrial bases, dense rail networks, and strategic transit positions. In contrast, most other EU nations fall well below the expected value line, with some—such as Luxembourg, Ireland, and Malta—contributing almost negligible volumes to the total.

The stark divergence indicates that rail freight activity in the EU is highly uneven, driven primarily by geography, economic scale, and infrastructure capacity. This observation aligns with the results of the chi-square goodness-of-fit test (p < 0.05), confirming that the distribution significantly deviates from equality.